In [ ]:
# Business Entity Resolution - cross-encoders on Kaggle GPUs. Settings: Accelerator = GPU T4 x2, Internet = ON
# One cell: prepare -> block -> kaggle_ce.py --stage all (ce1 = e5-base, ce2 = e5-large, ce4 = e5-large round 2)
import os, glob, shutil, subprocess, sys
# locate the uploaded dataset (folder containing code/ and dataset/)
roots = glob.glob('/kaggle/input/*')
print(roots)
ROOT = [r for r in roots if os.path.isdir(os.path.join(r, 'code'))][0]
CODE = os.path.join(ROOT, 'code')
DATA = glob.glob(os.path.join(ROOT, '**', 'train_ground_truth.tsv'), recursive=True)[0].rsplit('/train/', 1)[0]
print('CODE', CODE, 'DATA', DATA)
# outputs of an earlier session (attached with Add Input), comma separated; finished stages are skipped.
# Leave empty for the first session.
PREV = ''
WORK = '/kaggle/working/work'
os.makedirs(WORK, exist_ok=True)
shutil.copy(os.path.join(CODE, 'token_map.json'), WORK)
def run(cmd):
    print('>>', cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    assert r.returncode == 0, cmd
run(f'cd {CODE} && python prepare.py --data {DATA} --work {WORK} --workers 4')
run(f'cd {CODE} && python block.py --work {WORK} --split train --workers 4')
run(f'cd {CODE} && python block.py --work {WORK} --split test --workers 4')
prev = f' --prev {PREV}' if PREV else ''
run(f'cd {CODE} && python kaggle_ce.py --work {WORK} --data {DATA} --out /kaggle/working --stage all' + prev)
# keep only the result files in /kaggle/working (outputs are downloadable after the run)
shutil.rmtree(WORK, ignore_errors=True)
print(os.listdir('/kaggle/working'))
